<a href="https://colab.research.google.com/github/Penguinboy1990/Machine-Learning/blob/main/Midterm/MidtermAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Midterm Analysis

1. What challenges did you encounter when provisioning and executing machine learning workflows in a managed cloud environment (such as Colab Enterprise or Compute Engine) compared to local or standard free-tier notebooks?

It was a struggle even understanding how to do anything. The directions said to use Google Cloud Console, but I couldn't get that to transfer to a GitHub repository like being in Google Colab like we were using previously. So, I copied and ran everything like 3 times, which definitely ate some money. Figuring out where all the buttons and functions were was a real struggle. Even finding how much of the $50 I used was difficult to locate and not at all intuitive - and its slow to refresh, taking a few hours to a few days to reveal that the API has been billed. It also took a while to find how to stop the API, so despite closing the web browser, my money has been eaten up overnight. Using the instructions, I think I get it now. I still would have liked a video demonstration or images pointing to what these options look like and how to find them. Not even YouTube videos were helpful.

2. How did you identify and handle implicit missing values (such as -200 sentinel indicators) across the sensor columns, and how did your imputation strategy affect model stability?

I used df.replace(-200, np.nan), to change all of the values of -200 to #N/A errors in Excel. Over 90% of the "NMHC(GT)" column was -200, which means the data was unusable for practical means. Then I used an imputer using the median and most frequent strategy. This might have had an unexpected consequence if there was no most frequent value or if the median of any given column was skewed or bimodal. Otherwise, this didn't seem to affect the stability of the model.

3. Explain how the Scikit-Learn ColumnTransformer and Pipeline streamline the data preparation process and prevent data leakage across preprocessing and training steps in your cloud pipeline.

ColumnTransformer routes different columns to different preprocessing. Numeric and categorical columns need different treatment. Numeric variables or features go through StandardScaler while categorical ones go through OneHotEncoder, and the outputs are concatenated into one matrix. Without this, I would have to slice the DataFrame, transform each piece separately, and stitch them back together. The pipeline turns the steps into one estimator. The numeric and categorical pipelines bundle their steps, and full_pipeline wraps the whole preprocessor for a single fit/transform call to run everything in the correct order.
The pipeline.fit() calls fit_transform on each step using only the data passed in while pipeline.predict() calls only transform, reusing the statistics learned during training. So, testing data doesn't "leak" and affect the training data. The oneHotEncoder() makes categories seen only in test data encode as all zeros instead of raising an error, so the encoder never needs to see test data.

4. How did comparing StandardScaler versus MinMaxScaler impact your model training convergence, gradient descent optimization, and final evaluation metrics?

The StandardScaler allows faster training convergence than MinMaxScaler. It also allows larger alpha values for faster training times or gradient descent. So, I keps StandardScaler for the final evaluation metrics.

5. How does configuring SGDClassifier differ from standard closed-form solvers, and what advantages do probability outputs and evaluation metrics provide when assessing classification performance?

The SGDClassifier takes a loss type, alpha value and a random starting state (35). SGDClassifiers with loss='log_loss' changes the model to a Logistic Regression model which is optimized with Stochastic Gradient Descent. SGDClassifier is sensitive to feature scaling especially for large data sets, needs more tuning, uses incremental weight updates, and gives a slightly noisier approximation than a closed-form solver. This is ideal for dealing with large datasets that cannot fit into memory. Probabilities are great here because they allow us to set flexible thresholds. We could also take advantage of stratification with probabilities. A standard solver uses the whole dataset at every step and runs until they find the regularized optimum, which is not optimal for larger data sets.

6. Based on your regularized regression models (Ridge versus Lasso), how did adjusting the regularization hyperparameter affect feature selection and coefficient shrinkage, and what operational environmental insights do these findings reveal?

For the ridge model, the alpha didn't matter all too much when it came to accuracy or training time. However, there were often many coefficients. For the lasso model, the regularization hyperparameter was much more important to its final state, and reduced the amount of coefficents and features it used for its final model. That means that ridge models were generally more accurate, but will always need every parameter at least slightly. On the other hand, lasso models require a low regularization hyperparameter but make the results much easier to parse, and uses less computation to classify things.